## 测试集

In [4]:
import json
from pathlib import Path


def build_text(row):
    """Compose article fields; retain text-only input for saved splits/legacy data."""
    if not isinstance(row, dict):
        raise ValueError("数据必须是一个对象")
    if "title" in row or "content" in row:
        parts = []
        for field in ("title", "content"):
            value = row.get(field)
            if value is None:
                continue
            if not isinstance(value, str):
                raise ValueError(f"{field} 必须是字符串或 null")
            value = value.strip()
            if value:
                parts.append(f"{field}: {value}")
        if not parts:
            raise ValueError("title 和 content 不能同时为空")
        return "\n".join(parts)
    text = row.get("text")
    if not isinstance(text, str) or not text.strip():
        raise ValueError("需要非空 title/content，或兼容格式的非空字符串 text")
    return text.strip()


def load_jsonl(path):
    with path.open(encoding="utf-8-sig") as f:
        return [json.loads(line) for line in f if line.strip()]


# 按实际文件位置修改
test_path = Path("/mnt/cognitive_classification/data/test.jsonl")
prediction_path = Path("/mnt/cognitive_classification/outputs/test_predictions.jsonl")

test_rows = load_jsonl(test_path)
predictions = load_jsonl(prediction_path)

if not test_rows:
    raise ValueError("测试集为空")

if len(test_rows) != len(predictions):
    raise ValueError("测试集和预测结果的条数不同，无法直接逐条比较")

correct = 0

for i, (sample, prediction) in enumerate(
    zip(test_rows, predictions), start=1
):
    # 当前 predict.py 按输入顺序输出；再核对文本，防止对应错误
    if build_text(sample) != prediction["text"]:
        raise ValueError(f"第 {i} 条文本不一致，请检查文件和样本顺序")

    true_label = str(sample["label"]).strip()
    predicted_label = str(prediction["label"]).strip()

    if true_label == predicted_label:
        correct += 1

total = len(test_rows)
print(f"测试样本数：{total}")
print(f"预测正确数：{correct}")
print(f"准确率：{correct / total:.2%}")

测试样本数：26
预测正确数：13
准确率：50.00%


数据整理成csv

In [6]:
import csv
import json
from pathlib import Path


# 修改这里，指定输入、输出文件
input_path = Path("/mnt/cognitive_classification/outputs/test_predictions.jsonl")
output_path = Path("/mnt/cognitive_classification/outputs/test_predictions.csv")

# 读取 JSONL，每行转换成一个字典
with input_path.open(encoding="utf-8-sig") as f:
    rows = [json.loads(line) for line in f if line.strip()]

if not rows:
    raise ValueError("输入文件为空")

if any(not isinstance(row, dict) for row in rows):
    raise ValueError("每条数据必须是 JSON 对象")

# 收集所有字段，作为 CSV 表头
columns = list(dict.fromkeys(key for row in rows for key in row))

# 列表、字典类型的值转换为 JSON 字符串
for row in rows:
    for key, value in row.items():
        if isinstance(value, (list, dict)):
            row[key] = json.dumps(value, ensure_ascii=False)

# 使用 x 模式，避免覆盖已有文件
with output_path.open("x", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

print(f"转换完成：{len(rows)} 条数据")
print(f"保存位置：{output_path}")

转换完成：26 条数据
保存位置：/mnt/cognitive_classification/outputs/test_predictions.csv
